In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Georgetown run/intersection imagery"
)

print("Exists:", os.path.exists(BASE_DIR))
print("Is directory:", os.path.isdir(BASE_DIR))

Exists: True
Is directory: True


In [ ]:
import os
import re
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Georgetown run/intersection imagery"
)

puma_folders = sorted(
    [
        folder_name
        for folder_name in os.listdir(BASE_DIR)
        if re.fullmatch(r"260\d{4}", folder_name)
        and os.path.isdir(
            os.path.join(BASE_DIR, folder_name)
        )
    ]
)

counts = []

for pumaid in puma_folders:
    folder_path = os.path.join(
        BASE_DIR,
        pumaid
    )

    image_files = [
        filename
        for filename in os.listdir(folder_path)
        if filename.lower().endswith(
            (".jpg", ".jpeg")
        )
    ]

    counts.append(
        {
            "pumaid": pumaid,
            "n_images": len(image_files)
        }
    )

counts_df = pd.DataFrame(counts)

print("PUMA folders found:", len(counts_df))
print("Total images:", counts_df["n_images"].sum())

display(counts_df)

PUMA folders found: 9
Total images: 16446


,pumaid,n_images
0,2600100,1847
1,2600802,1789
2,2601200,1588
3,2601600,1602
4,2601701,1651
5,2601703,1993
6,2602903,1976
7,2603203,2000
8,2603212,2000


In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [ ]:
import os

print(
    os.listdir(
        "/content/drive/MyDrive/Georgetown run/intersection imagery/2602903"
    )[:10]
)

['92632.jpg', '461691.jpg', '471898.jpg', '233948.jpg', '121751.jpg', '287914.jpg', '294822.jpg', '505639.jpg', '392013.jpg', '107815.jpg']


In [ ]:
import glob
from PIL import Image

matches = glob.glob(
    "/content/drive/MyDrive/Georgetown run/"
    "intersection imagery/*/376384.*"
)

print(matches)

test_path = matches[0]

with Image.open(test_path) as img:
    img.load()
    print("Successfully read:", img.size, img.mode)

['/content/drive/MyDrive/Georgetown run/intersection imagery/2601600/376384.jpg']


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (134217728 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Successfully read: (16384, 8192) RGB


# run3

In [ ]:
import os
import asyncio
import re
import random
import time
from io import BytesIO

import pandas as pd
from PIL import Image, ImageOps

from google.colab import drive, userdata
from google import genai
from google.genai import types
from pydantic import BaseModel, Field


# ============================================================
# CONFIGURATION
# ============================================================

# IMPORTANT:
# Keep this False so your existing successful rows are preserved.
START_FRESH = False

MODEL_NAME = "gemini-3.1-flash-lite"

# Conservative settings for very large panorama files.
MAX_CONCURRENT_TASKS = 5
CHUNK_SIZE = 50

FILE_READ_RETRIES = 5
FILE_READ_TIMEOUT_SECONDS = 120

GEMINI_RETRIES = 4
GEMINI_TIMEOUT_SECONDS = 120

# Remaining images are resized in memory before being sent to Gemini.
# Original files in Google Drive are not modified.
RESIZE_MAX_SIDE = 4096
JPEG_QUALITY = 90


# ============================================================
# 1. MOUNT GOOGLE DRIVE AND DEFINE PATHS
# ============================================================

drive.mount("/content/drive")

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Georgetown run/intersection imagery"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Georgetown run"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_PATH = os.path.join(
    OUTPUT_DIR,
    "run3 detection.csv"
)


# ============================================================
# 2. VALIDATE CONFIGURATION
# ============================================================

if MODEL_NAME is None or not str(MODEL_NAME).strip():
    raise ValueError("MODEL_NAME must be set before launching Run 3.")

MODEL_NAME = str(MODEL_NAME).strip()

if MAX_CONCURRENT_TASKS < 1:
    raise ValueError("MAX_CONCURRENT_TASKS must be at least 1.")

if CHUNK_SIZE < 1:
    raise ValueError("CHUNK_SIZE must be at least 1.")

if FILE_READ_RETRIES < 1:
    raise ValueError("FILE_READ_RETRIES must be at least 1.")

if GEMINI_RETRIES < 1:
    raise ValueError("GEMINI_RETRIES must be at least 1.")

if RESIZE_MAX_SIDE < 1:
    raise ValueError("RESIZE_MAX_SIDE must be at least 1.")

if not 1 <= JPEG_QUALITY <= 100:
    raise ValueError("JPEG_QUALITY must be between 1 and 100.")


# ============================================================
# 3. OPTIONAL CLEAN RESTART
# ============================================================

if START_FRESH:
    print(
        "🗑️ START_FRESH is True. "
        "Deleting any existing run3 detection.csv..."
    )

    if os.path.exists(CSV_PATH):
        try:
            os.remove(CSV_PATH)
            print("✅ Existing run3 detection.csv deleted.")
            time.sleep(2)
        except Exception as error:
            raise RuntimeError(
                "Could not delete the existing Run 3 output:\n"
                f"{CSV_PATH}\n\n"
                f"Error: {type(error).__name__}: {error}"
            )
    else:
        print(
            "✅ No existing run3 detection.csv found. "
            "Starting with a clean slate."
        )


# ============================================================
# 4. CONFIRM IMAGE SOURCE IS ACCESSIBLE
# ============================================================

print("🔄 Waking up Google Drive file system...")
os.listdir("/content/drive/MyDrive")

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        "The Run 3 image directory could not be found:\n"
        f"{BASE_DIR}\n\n"
        "Confirm the folder path matches your Drive setup."
    )

try:
    os.listdir(BASE_DIR)
except Exception as error:
    raise RuntimeError(
        "The Run 3 image directory exists but could not be read:\n"
        f"{BASE_DIR}\n\n"
        f"Error: {type(error).__name__}: {error}"
    )

if not START_FRESH and not os.path.exists(CSV_PATH):
    print(
        "⚠️ START_FRESH is False, but no existing "
        "run3 detection.csv was found."
    )
    print("The script will process all available Run 3 images.")

print("✅ Run 3 image directory successfully found!")
print(f"📁 Image source: {BASE_DIR}")
print(f"📄 Output CSV: {CSV_PATH}")
print(f"🤖 Selected model: {MODEL_NAME}")
print(
    "🖼️ Preprocessing for remaining images: "
    f"long side ≤ {RESIZE_MAX_SIDE}px, JPEG quality {JPEG_QUALITY}"
)
print(f"⚙️ Concurrency: {MAX_CONCURRENT_TASKS}")
print(f"💾 Checkpoint size: {CHUNK_SIZE}")


# ============================================================
# 5. INITIALIZE GEMINI CLIENT
# ============================================================

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY was not found in Colab Secrets.")

client = genai.Client(api_key=api_key)
aclient = client.aio


# ============================================================
# 6. DEFINE THE 17 RUBRIC CATEGORIES
# ============================================================

CATEGORIES = [
    "Overgrown tall weeds and unmanaged grass.",
    "Plywood boards covering windows/doors.",
    "Shattered glass or broken window panes.",
    "Peeling paint or chipped wood on siding/porches.",
    "Missing siding panels and exposed house wrap.",
    "Crumbling concrete porch steps and leaning porch roofs.",
    "Weathered asphalt cracks in the street.",
    "Potholes or severely deteriorated road surface.",
    "Colorful graffiti on walls.",
    "Loose litter or scattered garbage piles.",
    "Wood fence or rusted chain-link fences.",
    "Metal security gate or bars over a residential door.",
    "Liquor, tobacco/vape, or dollar-store retail visible.",
    (
        "Visibly abandoned or inoperable vehicles, such as "
        "flat tires, missing parts, broken windows, or "
        "long-term immobility signs."
    ),
    (
        "Damaged, missing, leaning, or poorly maintained "
        "public fixtures, such as street signs, utility poles, "
        "bus stops, or streetlights."
    ),
    "Residential garage or garage door visible.",
    "Window air-conditioning unit visible."
]


# ============================================================
# 7. STRUCTURED OUTPUT SCHEMA
# ============================================================

class TriageResult(BaseModel):
    overgrown_weeds: bool = Field(
        description="Overgrown tall weeds and unmanaged grass."
    )

    plywood_boards: bool = Field(
        description="Plywood boards covering windows/doors."
    )

    shattered_glass: bool = Field(
        description="Shattered glass or broken window panes."
    )

    peeling_paint: bool = Field(
        description="Peeling paint or chipped wood on siding/porches."
    )

    missing_siding: bool = Field(
        description="Missing siding panels and exposed house wrap."
    )

    crumbling_concrete: bool = Field(
        description=(
            "Crumbling concrete porch steps and leaning porch roofs."
        )
    )

    asphalt_cracks: bool = Field(
        description="Weathered asphalt cracks in the street."
    )

    potholes: bool = Field(
        description="Potholes or severely deteriorated road surface."
    )

    graffiti: bool = Field(
        description="Colorful graffiti on walls."
    )

    loose_litter: bool = Field(
        description="Loose litter or scattered garbage piles."
    )

    wood_or_chain_fence: bool = Field(
        description="Wood fence or rusted chain-link fences."
    )

    security_gate: bool = Field(
        description=(
            "Metal security gate or bars over a residential door."
        )
    )

    retail_signs: bool = Field(
        description=(
            "Liquor, tobacco/vape, or dollar-store retail visible."
        )
    )

    abandoned_vehicles: bool = Field(
        description=(
            "Visibly abandoned or inoperable vehicles, such as "
            "flat tires, missing parts, broken windows, or "
            "long-term immobility signs."
        )
    )

    damaged_fixtures: bool = Field(
        description=(
            "Damaged, missing, leaning, or poorly maintained public "
            "fixtures, such as street signs, utility poles, bus stops, "
            "or streetlights."
        )
    )

    garage_visible: bool = Field(
        description="Residential garage or garage door visible."
    )

    window_ac: bool = Field(
        description="Window air-conditioning unit visible."
    )


# ============================================================
# 8. MAP SCHEMA KEYS TO EXACT CSV COLUMN NAMES
# ============================================================

SCHEMA_KEY_MAP = {
    "overgrown_weeds":
        "Overgrown tall weeds and unmanaged grass.",

    "plywood_boards":
        "Plywood boards covering windows/doors.",

    "shattered_glass":
        "Shattered glass or broken window panes.",

    "peeling_paint":
        "Peeling paint or chipped wood on siding/porches.",

    "missing_siding":
        "Missing siding panels and exposed house wrap.",

    "crumbling_concrete":
        "Crumbling concrete porch steps and leaning porch roofs.",

    "asphalt_cracks":
        "Weathered asphalt cracks in the street.",

    "potholes":
        "Potholes or severely deteriorated road surface.",

    "graffiti":
        "Colorful graffiti on walls.",

    "loose_litter":
        "Loose litter or scattered garbage piles.",

    "wood_or_chain_fence":
        "Wood fence or rusted chain-link fences.",

    "security_gate":
        "Metal security gate or bars over a residential door.",

    "retail_signs":
        "Liquor, tobacco/vape, or dollar-store retail visible.",

    "abandoned_vehicles": (
        "Visibly abandoned or inoperable vehicles, such as "
        "flat tires, missing parts, broken windows, or "
        "long-term immobility signs."
    ),

    "damaged_fixtures": (
        "Damaged, missing, leaning, or poorly maintained public "
        "fixtures, such as street signs, utility poles, bus stops, "
        "or streetlights."
    ),

    "garage_visible":
        "Residential garage or garage door visible.",

    "window_ac":
        "Window air-conditioning unit visible."
}


# ============================================================
# 9. EXACT MODEL PROMPT — UNCHANGED
# ============================================================

PROMPT = """
You are flagging visible cues in street-level panorama images.
Evaluate the image against the 17 indicators requested in the JSON schema.
Only select from the 17 indicators. Flag an indicator only if it is visible in the image.
Do not add any indicators outside the given 17. Do not make assumptions.
If the item is clearly visible, set its corresponding value to true. Otherwise, set it to false.
"""


# ============================================================
# 10. FIXED CSV STRUCTURE
# ============================================================

OUTPUT_COLUMNS = [
    "folder_id",
    "image_id",
    "status",
    *SCHEMA_KEY_MAP.values(),
    "error"
]


def empty_result_row(folder_id, image_id, status, error=None):
    result = {
        "folder_id": str(folder_id),
        "image_id": str(image_id),
        "status": status
    }

    for csv_header in SCHEMA_KEY_MAP.values():
        result[csv_header] = None

    result["error"] = error

    return result


# ============================================================
# 11. IMAGE-ID HELPER
# ============================================================

def get_image_id(image_filename):
    image_id = re.sub(r"\D", "", image_filename)

    if not image_id:
        raise ValueError(
            "No numeric image ID could be extracted from filename: "
            f"{image_filename}"
        )

    return image_id


# ============================================================
# 12. IMAGE PREPROCESSING HELPER
# ============================================================

# These are trusted project images.
# Disable Pillow's unusually-large-image warning threshold.
Image.MAX_IMAGE_PIXELS = None


def load_and_resize_image_bytes(
    file_path,
    max_side=RESIZE_MAX_SIDE,
    quality=JPEG_QUALITY
):
    """
    Read one image from Drive, resize it in memory while preserving
    aspect ratio, and return JPEG bytes.

    The original Drive image is not modified.
    """

    with Image.open(file_path) as image:
        original_format = image.format

        # Request reduced-resolution JPEG decoding when supported.
        # This is useful for very large 16384 x 8192 panoramas.
        if original_format == "JPEG":
            image.draft(
                "RGB",
                (max_side, max_side)
            )

        image = ImageOps.exif_transpose(image)

        if image.mode != "RGB":
            image = image.convert("RGB")

        image.thumbnail(
            (max_side, max_side),
            Image.Resampling.LANCZOS
        )

        output_buffer = BytesIO()

        image.save(
            output_buffer,
            format="JPEG",
            quality=quality,
            optimize=True
        )

        image_bytes = output_buffer.getvalue()

    if not image_bytes:
        raise OSError(
            "The preprocessed image returned zero bytes."
        )

    return image_bytes


# ============================================================
# 13. WORKER COROUTINE
# ============================================================

async def analyze_image(
    semaphore,
    folder_id,
    image_id,
    file_path
):
    async with semaphore:
        await asyncio.sleep(
            random.uniform(0.1, 0.3)
        )

        # ----------------------------------------------------
        # Read and resize image with retries and timeout
        # ----------------------------------------------------

        image_bytes = None

        for read_attempt in range(FILE_READ_RETRIES):
            try:
                image_bytes = await asyncio.wait_for(
                    asyncio.to_thread(
                        load_and_resize_image_bytes,
                        file_path,
                        RESIZE_MAX_SIDE,
                        JPEG_QUALITY
                    ),
                    timeout=FILE_READ_TIMEOUT_SECONDS
                )

                break

            except asyncio.TimeoutError:
                has_retry_remaining = (
                    read_attempt < FILE_READ_RETRIES - 1
                )

                if has_retry_remaining:
                    wait_time = (
                        (2 ** read_attempt)
                        + random.uniform(0.5, 1.5)
                    )

                    print(
                        f"⚠️ Image preprocessing timeout on {image_id}. "
                        f"Retry {read_attempt + 1}/{FILE_READ_RETRIES} "
                        f"in {wait_time:.1f}s..."
                    )

                    await asyncio.sleep(wait_time)
                    continue

                return empty_result_row(
                    folder_id=folder_id,
                    image_id=image_id,
                    status="FAILED",
                    error=(
                        "TimeoutError: image read/resizing exceeded "
                        f"{FILE_READ_TIMEOUT_SECONDS} seconds after "
                        f"{FILE_READ_RETRIES} attempts."
                    )
                )

            except OSError as error:
                has_retry_remaining = (
                    read_attempt < FILE_READ_RETRIES - 1
                )

                if has_retry_remaining:
                    wait_time = (
                        (2 ** read_attempt)
                        + random.uniform(0.5, 1.5)
                    )

                    print(
                        f"⚠️ Image read error on {image_id}. "
                        f"Retry {read_attempt + 1}/{FILE_READ_RETRIES} "
                        f"in {wait_time:.1f}s..."
                    )

                    await asyncio.sleep(wait_time)
                    continue

                return empty_result_row(
                    folder_id=folder_id,
                    image_id=image_id,
                    status="FAILED",
                    error=(
                        "Failed to read/resize file after "
                        f"{FILE_READ_RETRIES} attempts: "
                        f"{type(error).__name__}: {error}"
                    )
                )

            except Exception as error:
                return empty_result_row(
                    folder_id=folder_id,
                    image_id=image_id,
                    status="FAILED",
                    error=(
                        "Unexpected image preprocessing error: "
                        f"{type(error).__name__}: {error}"
                    )
                )

        if image_bytes is None:
            return empty_result_row(
                folder_id=folder_id,
                image_id=image_id,
                status="FAILED",
                error="Preprocessed image bytes were not loaded."
            )

        # ----------------------------------------------------
        # Build Gemini image input
        # ----------------------------------------------------

        image_part = types.Part.from_bytes(
            data=image_bytes,
            mime_type="image/jpeg"
        )

        # ----------------------------------------------------
        # Call Gemini with retries and timeout
        # ----------------------------------------------------

        for attempt in range(GEMINI_RETRIES):
            try:
                response = await asyncio.wait_for(
                    aclient.models.generate_content(
                        model=MODEL_NAME,
                        contents=[
                            PROMPT,
                            image_part
                        ],
                        config=types.GenerateContentConfig(
                            response_mime_type="application/json",
                            response_json_schema=(
                                TriageResult.model_json_schema()
                            )
                        )
                    ),
                    timeout=GEMINI_TIMEOUT_SECONDS
                )

                if not response.text:
                    raise ValueError(
                        "The model returned an empty response."
                    )

                parsed = TriageResult.model_validate_json(
                    response.text
                )

                parsed_data = parsed.model_dump()

                result = empty_result_row(
                    folder_id=folder_id,
                    image_id=image_id,
                    status="SUCCESS",
                    error=None
                )

                for schema_key, csv_header in SCHEMA_KEY_MAP.items():
                    result[csv_header] = (
                        1 if parsed_data.get(schema_key) else 0
                    )

                return result

            except asyncio.TimeoutError:
                has_retry_remaining = (
                    attempt < GEMINI_RETRIES - 1
                )

                if has_retry_remaining:
                    wait_time = (
                        (2 ** attempt)
                        + random.uniform(1, 3)
                    )

                    print(
                        f"⚠️ Model timeout on {image_id}. "
                        f"Retry {attempt + 1}/{GEMINI_RETRIES} "
                        f"in {wait_time:.1f}s..."
                    )

                    await asyncio.sleep(wait_time)
                    continue

                full_error = (
                    "TimeoutError: Model request exceeded "
                    f"{GEMINI_TIMEOUT_SECONDS} seconds after "
                    f"{GEMINI_RETRIES} attempts."
                )

                print(
                    f"\n❌ Failed image {image_id}:\n"
                    f"{full_error}\n"
                )

                return empty_result_row(
                    folder_id=folder_id,
                    image_id=image_id,
                    status="FAILED",
                    error=full_error
                )

            except Exception as error:
                error_text = str(error).upper()

                transient_markers = [
                    "429",
                    "500",
                    "502",
                    "503",
                    "504",
                    "RESOURCE_EXHAUSTED",
                    "RATE",
                    "QUOTA",
                    "LIMIT",
                    "OVERLOADED",
                    "UNAVAILABLE",
                    "TIMEOUT",
                    "TIMED OUT",
                    "CONNECTION",
                    "CONNECTION RESET",
                    "INTERNAL"
                ]

                is_transient = any(
                    marker in error_text
                    for marker in transient_markers
                )

                has_retry_remaining = (
                    attempt < GEMINI_RETRIES - 1
                )

                if is_transient and has_retry_remaining:
                    wait_time = (
                        (2 ** attempt)
                        + random.uniform(1, 3)
                    )

                    print(
                        f"⚠️ API error on {image_id}. "
                        f"Retry {attempt + 1}/{GEMINI_RETRIES} "
                        f"in {wait_time:.1f}s..."
                    )

                    await asyncio.sleep(wait_time)
                    continue

                full_error = (
                    f"{type(error).__name__}: {repr(error)}"
                )

                print(
                    f"\n❌ Failed image {image_id}:\n"
                    f"{full_error}\n"
                )

                return empty_result_row(
                    folder_id=folder_id,
                    image_id=image_id,
                    status="FAILED",
                    error=full_error
                )

        return empty_result_row(
            folder_id=folder_id,
            image_id=image_id,
            status="FAILED",
            error="Maximum model retries exceeded."
        )


# ============================================================
# 14. SAVE OR UPDATE CHECKPOINT
# ============================================================

def save_checkpoint(batch_results):
    new_df = pd.DataFrame(
        batch_results
    ).reindex(
        columns=OUTPUT_COLUMNS
    )

    new_df["folder_id"] = (
        new_df["folder_id"].astype(str)
    )

    new_df["image_id"] = (
        new_df["image_id"].astype(str)
    )

    if os.path.exists(CSV_PATH):
        existing_df = pd.read_csv(
            CSV_PATH,
            on_bad_lines="skip",
            dtype={
                "folder_id": str,
                "image_id": str
            }
        ).reindex(
            columns=OUTPUT_COLUMNS
        )

        combined_df = pd.concat(
            [
                new_df,
                existing_df
            ],
            ignore_index=True
        )

        # New results replace old rows for the same image.
        # This lets a retried SUCCESS replace an earlier FAILED row.
        combined_df = combined_df.drop_duplicates(
            subset=[
                "folder_id",
                "image_id"
            ],
            keep="first"
        )

    else:
        combined_df = new_df

    combined_df = combined_df.sort_values(
        [
            "folder_id",
            "image_id"
        ]
    ).reset_index(
        drop=True
    )

    temporary_path = CSV_PATH + ".tmp"

    combined_df.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        CSV_PATH
    )

    return combined_df


# ============================================================
# 15. MAIN ASYNC COORDINATOR
# ============================================================

async def main():
    processed_images = set()

    # --------------------------------------------------------
    # Load existing successful Run 3 rows
    # --------------------------------------------------------

    if os.path.exists(CSV_PATH) and not START_FRESH:
        try:
            existing_df = pd.read_csv(
                CSV_PATH,
                on_bad_lines="skip",
                dtype={
                    "folder_id": str,
                    "image_id": str
                }
            )

            required_checkpoint_columns = {
                "folder_id",
                "image_id",
                "status"
            }

            missing_checkpoint_columns = (
                required_checkpoint_columns
                - set(existing_df.columns)
            )

            if missing_checkpoint_columns:
                raise ValueError(
                    "Existing CSV is missing required columns: "
                    f"{sorted(missing_checkpoint_columns)}"
                )

            existing_df["folder_id"] = (
                existing_df["folder_id"].astype(str)
            )

            existing_df["image_id"] = (
                existing_df["image_id"].astype(str)
            )

            success_df = existing_df[
                existing_df["status"].eq("SUCCESS")
            ].copy()

            processed_images = set(
                zip(
                    success_df["folder_id"],
                    success_df["image_id"]
                )
            )

            print(
                f"🔄 Checkpoint loaded: "
                f"{len(processed_images):,} images "
                "successfully completed."
            )

            print(
                "\nExisting checkpoint status counts:"
            )

            print(
                existing_df["status"].value_counts(
                    dropna=False
                )
            )

        except Exception as error:
            raise RuntimeError(
                "Could not safely load the existing "
                "Run 3 checkpoint:\n"
                f"{type(error).__name__}: {error}"
            )

    # --------------------------------------------------------
    # Find targeted PUMA subfolders
    # --------------------------------------------------------

    subfolders = sorted(
        [
            folder_name
            for folder_name in os.listdir(BASE_DIR)
            if re.fullmatch(
                r"260\d{4}",
                folder_name
            )
            and os.path.isdir(
                os.path.join(
                    BASE_DIR,
                    folder_name
                )
            )
        ]
    )

    print(
        f"\n📂 Found {len(subfolders)} PUMA subfolders:"
    )

    for folder_name in subfolders:
        print(folder_name)

    if len(subfolders) != 9:
        raise ValueError(
            f"Expected 9 PUMA folders, "
            f"but found {len(subfolders)}:\n"
            f"{subfolders}"
        )

    # --------------------------------------------------------
    # Gather images and validate identifiers
    # --------------------------------------------------------

    all_image_tasks = []
    all_image_keys = set()

    total_images_found = 0
    skipped_successful = 0
    puma_counts = []

    for folder_id in subfolders:
        folder_path = os.path.join(
            BASE_DIR,
            folder_id
        )

        image_files = sorted(
            [
                filename
                for filename in os.listdir(folder_path)
                if filename.lower().endswith(
                    (
                        ".jpg",
                        ".jpeg"
                    )
                )
            ]
        )

        puma_counts.append(
            {
                "folder_id": folder_id,
                "n_images": len(image_files)
            }
        )

        total_images_found += len(image_files)

        for image_filename in image_files:
            image_id = get_image_id(
                image_filename
            )

            key = (
                str(folder_id),
                str(image_id)
            )

            if key in all_image_keys:
                raise ValueError(
                    "Duplicate folder_id + image_id detected:\n"
                    f"folder_id={folder_id}, "
                    f"image_id={image_id}"
                )

            all_image_keys.add(key)

            if (
                not START_FRESH
                and key in processed_images
            ):
                skipped_successful += 1
                continue

            file_path = os.path.join(
                folder_path,
                image_filename
            )

            all_image_tasks.append(
                (
                    folder_id,
                    image_id,
                    file_path
                )
            )

    total_tasks = len(all_image_tasks)

    print("\nImage counts by PUMA:")

    print(
        pd.DataFrame(
            puma_counts
        ).to_string(
            index=False
        )
    )

    print(
        f"\n🖼️ Total Run 3 images found: "
        f"{total_images_found:,}"
    )

    print(
        f"✅ Previously successful images skipped: "
        f"{skipped_successful:,}"
    )

    print(
        "🚀 Images remaining for processing or retry: "
        f"{total_tasks:,}"
    )

    if total_images_found == 0:
        raise ValueError(
            "No images found in targeted subfolders."
        )

    if total_tasks == 0:
        print(
            "\n🎉 All images are already marked SUCCESS."
        )
        return

    # --------------------------------------------------------
    # Batch processing loop
    # --------------------------------------------------------

    semaphore = asyncio.Semaphore(
        MAX_CONCURRENT_TASKS
    )

    total_batches = (
        total_tasks
        + CHUNK_SIZE
        - 1
    ) // CHUNK_SIZE

    for start_index in range(
        0,
        total_tasks,
        CHUNK_SIZE
    ):
        batch_number = (
            start_index // CHUNK_SIZE
        ) + 1

        chunk = all_image_tasks[
            start_index:
            start_index + CHUNK_SIZE
        ]

        print(
            f"\n⚙️ Processing Run 3 batch "
            f"{batch_number} / {total_batches}..."
        )

        tasks = [
            analyze_image(
                semaphore,
                folder_id,
                image_id,
                file_path
            )
            for (
                folder_id,
                image_id,
                file_path
            ) in chunk
        ]

        batch_results = await asyncio.gather(
            *tasks
        )

        combined_df = save_checkpoint(
            batch_results
        )

        print(
            "💾 Checkpoint saved. Current Run 3 progress: "
            f"{len(combined_df):,} / "
            f"{total_images_found:,}"
        )

        print("\nBatch statuses:")

        print(
            pd.Series(
                [
                    row["status"]
                    for row in batch_results
                ],
                name="status"
            ).value_counts(
                dropna=False
            )
        )

        print(
            "\nRows currently stored in Run 3 CSV: "
            f"{len(combined_df):,}"
        )

    # --------------------------------------------------------
    # Final verification
    # --------------------------------------------------------

    print(
        "\n🎉 Run 3 processing finished."
    )

    print(
        f"\nFinal output:\n{CSV_PATH}"
    )

    final_df = pd.read_csv(
        CSV_PATH,
        dtype={
            "folder_id": str,
            "image_id": str
        },
        on_bad_lines="skip"
    )

    print(
        "\nFinal status counts:"
    )

    print(
        final_df["status"].value_counts(
            dropna=False
        )
    )

    duplicate_count = final_df.duplicated(
        subset=[
            "folder_id",
            "image_id"
        ]
    ).sum()

    print(
        "\nDuplicate folder_id + image_id rows: "
        f"{duplicate_count:,}"
    )

    success_count = final_df[
        "status"
    ].eq(
        "SUCCESS"
    ).sum()

    if (
        len(final_df) == total_images_found
        and success_count == total_images_found
        and duplicate_count == 0
    ):
        print(
            "\n✅ Perfect final check: "
            "every image is SUCCESS."
        )
    else:
        print(
            "\n⚠️ Final output is not yet perfect. "
            "Rerun with START_FRESH = False "
            "to retry remaining failures."
        )


# ============================================================
# EXECUTE IN GOOGLE COLAB
# ============================================================

await main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Waking up Google Drive file system...
✅ Run 3 image directory successfully found!
📁 Image source: /content/drive/MyDrive/Georgetown run/intersection imagery
📄 Output CSV: /content/drive/MyDrive/Georgetown run/run3 detection.csv
🤖 Selected model: gemini-3.1-flash-lite
🖼️ Preprocessing for remaining images: long side ≤ 4096px, JPEG quality 90
⚙️ Concurrency: 5
💾 Checkpoint size: 50
🔄 Checkpoint loaded: 6,050 images successfully completed.

Existing checkpoint status counts:
status
SUCCESS    6050
Name: count, dtype: int64

📂 Found 9 PUMA subfolders:
2600100
2600802
2601200
2601600
2601701
2601703
2602903
2603203
2603212

Image counts by PUMA:
folder_id  n_images
  2600100      1847
  2600802      1789
  2601200      1588
  2601600      1602
  2601701      1651
  2601703      1993
  2602903      1976
  2603203      2000
  2603212      2000

🖼️ Total Run 3 imag